# ADS Bibliometric Analysis Pipeline

This notebook is the single entry point for the `ads_bib` package.
It executes all steps sequentially, with configuration cells between steps
so that downstream parameters can be set based on upstream results.

**Pipeline Steps:**
1. Search & Export â€” Query ADS API, resolve bibcodes to metadata
2. Translation â€” Detect languages, translate non-English titles/abstracts
3. Tokenization â€” Create full_text, tokenize with spaCy
4. AND â€” Author Name Disambiguation (placeholder)
5. Topic Modeling & Curation â€” BERTopic + datamapplot + cluster removal
6. Citation Networks â€” Direct, Co-Citation, Bibliographic Coupling, Author Co-Citation

---
## Setup

In [1]:
import os
from pathlib import Path

# Initialize paths (relative to this notebook's location)
from ads_bib.config import init_paths, load_env
from ads_bib.run_manager import RunManager
from ads_bib._utils.costs import CostTracker

paths = init_paths()  # uses CWD = notebook directory
load_env()

# Cost tracker for all API calls (OpenRouter, etc.)
tracker = CostTracker()

# --- INITIALIZE RUN ---
# This tracks all parameters and saves outputs to a unique folder
run = RunManager(run_name='ADS_Curation_Run')

In [2]:
import logging

root_logger = logging.getLogger()
root_logger.setLevel(logging.INFO)
formatter = logging.Formatter("%(message)s")

if root_logger.handlers:
    for handler in root_logger.handlers:
        handler.setLevel(logging.INFO)
        handler.setFormatter(formatter)
else:
    handler = logging.StreamHandler()
    handler.setLevel(logging.INFO)
    handler.setFormatter(formatter)
    root_logger.addHandler(handler)

logger = logging.getLogger("pipeline")

---
## Global Run Control

Set run-level switches here. Phase-specific parameters are configured in each phase section below.


In [3]:
import random
import numpy as np

# ── CONFIGURE ─────────────────────────────────────────
# 0 = Run everything (Search to End)
# 4 = Load processed data (Skip Search, Translate, Tokenize) -> Start at Topic Modeling
START_AT_PHASE = 0
FORCE_REFRESH = True

# Deterministic seed for notebook-side randomness (sampling + reduction params)
RANDOM_SEED = 42

# Shared OpenRouter pricing mode (used in translation, embeddings, topic labeling)
OPENROUTER_COST_MODE = "hybrid"  # 'hybrid', 'strict', 'fast'
# ──────────────────────────────────────────────────────

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

---
# Phase 1: Search & Export

Query the NASA ADS API for publications matching a search string,
then resolve all bibcodes (publications + references) into full metadata.

### 1.1 Search Configuration

Set query, years, and retrieval limits for your research question.
These values define the corpus scope for all downstream steps.


In [4]:
# --- SEARCH CONFIGURATION ---
# Modify the query string for your research question.
# See: https://ui.adsabs.harvard.edu/help/search/search-syntax

ADS_TOKEN = os.getenv("ADS_API_KEY")

Set_A = "docs(library/P0hyiLw0T8qsyHSBpWigJA)"
Set_B = f"citations({Set_A}) AND year:1915-2000"
Set_C = f"citations(citations({Set_A})) AND year:1915-2000"
Set_D = f"({Set_A}) OR ({Set_B}) OR ({Set_C})"
Set_T = "(docs(library/_-RCjJotRKCZC1PP03MqwA) AND year:1915-2000)"

gravity_relativity_terms = [
    '(general AND relativi*)', 'gravit*', '(allgemein* AND relativi*)',
    '"relativité générale"', '"teoria della relatività"',
    '"Gravité quantique"', '"Gravità quantistica"',
    '(einheitlich* AND feld*)', 'Quantengravitation', '"champ unifié"',
    '(unified AND field*)', '"quantum gravity"', 'cosmo*', 'Kosmo*',
]
Set_E = f"abs:({' OR '.join(gravity_relativity_terms)}) AND year:1915-2000"

# Example quick focus query
QUERY = 'author:"Treder, H*"'
# Alternative broader query:
# QUERY = f"({Set_D}) OR ({Set_T}) OR ({Set_E})"

In [5]:
# # ...existing code...
# # --- SEARCH CONFIGURATION ---
# # Modify the query string for your research question.
# # See: https://ui.adsabs.harvard.edu/help/search/search-syntax

# ADS_TOKEN = os.getenv("ADS_API_KEY")

# # 1. Historischer Seed: Einsteins Publikationen (1911-1955)
# Set_A = "docs(library/P0hyiLw0T8qsyHSBpWigJA)"

# # 2. Direkte Rezeption: 1-Hop Zitationen ab 1915
# Set_B = f"citations({Set_A}) AND year:1915-2000"

# # 3. Strukturelle Anker-Begriffe (Mehrsprachig: EN, DE, FR, IT)
# generic_anchors = [
#     'relativ*', 'einstein*', 
#     'spacetime', '"space-time"', 'raumzeit', '"espace-temps"', 'spaziotempo', '"spazio-tempo"',
#     'tensor*', 'tenseur*', 
#     'metric*', 'metrik*', 'metriqu*', 
#     'curvature', 'krümmung', 'kruemmung', 'courbure', 'curvatura',
#     'geodesic*', 'geodät*', 'geodaet*', 'geodesiqu*', 'geodetic*'
# ]

# # 4. Indirekte Rezeption: 2-Hop Zitationen ab 1915 (Gefiltert durch Anker)
# Set_C = f"citations(citations({Set_A})) AND year:1915-2000 AND abs:({' OR '.join(generic_anchors)})"

# # 5. Treder Library
# Set_T = "(docs(library/_-RCjJotRKCZC1PP03MqwA) AND year:1915-2000)"

# # 6. Text-Suche: Core-Begriffe und verankerte breite Begriffe
# grg_core_terms = [
#     '(general AND relativi*)', '(allgemein* AND relativi*)',
#     '"relativité générale"', '"relatività generale"',
#     '"Gravité quantique"', '"Gravità quantistica"', 'Quantengravitation', '"quantum gravity"',
#     '(einheitlich* AND feld*)', '"champ unifié"', '(unified AND field*)'
# ]

# broad_terms = ['gravit*', 'cosmo*', 'Kosmo*']
# anchored_broad = f"({' OR '.join(broad_terms)}) AND ({' OR '.join(generic_anchors)})"

# Set_E = f"abs:({' OR '.join(grg_core_terms)} OR ({anchored_broad})) AND year:1911-2000"

# # Finale Query
# QUERY = f"({Set_A}) OR ({Set_B}) OR ({Set_C}) OR ({Set_T}) OR ({Set_E})"

### 1.2 Execute Search

Run the ADS search and persist bibcodes/results to this run folder.
This creates a reproducible input snapshot for later phases.


In [6]:
if START_AT_PHASE <= 1:
    from ads_bib.search import search_ads

    bibcodes, references, esources, fulltext_urls = search_ads(
        QUERY, ADS_TOKEN, raw_dir=paths["raw"], force_refresh=FORCE_REFRESH,
    )
else:
    logger.info("Skipping Phase 1 Search (START_AT_PHASE=%s)", START_AT_PHASE)

Starting ADS search ...
  382 records fetched ...
Done. Bibcodes: 382 | Unique refs: 499 | Total refs: 796
Saved: search_results_20260224_123027.pkl
Latest: search_results_latest.pkl


### 1.3 Export & Resolve Metadata

Resolve publications and references to structured metadata tables.
This normalizes schemas before translation/tokenization/topic modeling.


In [7]:
if START_AT_PHASE <= 1:
    from ads_bib.export import resolve_dataset

    publications, refs = resolve_dataset(
        bibcodes, references, esources, fulltext_urls, ADS_TOKEN,
        cache_dir=paths["raw"], force_refresh=FORCE_REFRESH,
    )
else:
    logger.info("Skipping Phase 1 Export (START_AT_PHASE=%s)", START_AT_PHASE)

=== Exporting publications ===
Exporting 382 bibcodes in 1 chunks (5 workers) ...


Exporting:   0%|          | 0/382 [00:00<?, ?it/s]

Publications: 382 records
=== Exporting references (499 unique) ===
Exporting 499 bibcodes in 1 chunks (5 workers) ...


Exporting:   0%|          | 0/499 [00:00<?, ?it/s]

References: 499 records


In [8]:
if START_AT_PHASE <= 1:
    display(publications.head(50))

,Bibcode,Author,Title,Year,Journal,Journal Abbreviation,Issue,Volume,First Page,Last Page,Abstract,Keywords,DOI,Affiliation,Category,Citation Count,References,PDF_URL
0,1971grun.book.....T,"[Treder, H.J.]",Gravitationstheorie und Äquivalenzprinzip.,1971,Gravitationstheorie und Äquivalenzprinzip,grun.book,,,,,,,,AA(-),book,52,[],None
1,1957AnP...454..369T,"[Treder, H.]",Stromladungsdefinition und elektrische Kraft i...,1957,Annalen der Physik,AnP,6,454,369,380,"Wie Infeld 1950 gezeigt hat, ergibt sich aus d...",,10.1002/andp.19574540614,AA(-),article,34,[1953PhRv...92.1567C],None
2,1967AnP...475..194T,"[Treder, HansJürgen]",Das makroskopische Gravitationsfeld in der ein...,1967,Annalen der Physik,AnP,3,475,194,206,A theory of the gravitation field is proposed ...,,10.1002/andp.19674750308,AA(-),article,38,"[1916AnP...354..769E, 1940PhRv...57..147R, 195...",None
3,1983AN....304..145T,"[Treder, H.J.]",The problem of the Grand Unification Theory,1983,Astronomische Nachrichten,AN,4,304,145,151,The evolution and fundamental questions of phy...,"Astronomical Models, Cosmology, Unified Field ...",10.1002/asna.2113040402,"AA(German Academy of Sciences, Potsdam)",article,1,[1950sts..book.....S],https://ui.adsabs.harvard.edu/link_gateway/198...
4,1972drdt.book.....T,"[Treder, H.J.]",Die Relativität der Trägheit.,1972,Die Relativität der Trägheit,drdt.book,,,,,,,,AA(-),book,31,[],None
5,1997GReGr..29..455V,"[von Borzeszkowski, HorstHeino, Treder, HansJü...",The Weyl-Cartan Space Problem in Purely Affine...,1997,General Relativity and Gravitation,GReGr,4,29,455,466,"According to Poincaré, only the ``epistemologi...",,10.1023/A:1018830631884,AA(-); AB(-),article,21,"[1950sts..book.....S, 1971GReGr...2..313T, 199...",None
6,1981AN....302..275T,"[Treder, H. J., Jackisch, G.]",On Soldners Value of Newtonian Deflection of L...,1981,Astronomische Nachrichten,AN,6,302,275,,,,10.1002/asna.2103020603,AA(-); AB(-),article,6,"[1970AnP...480..315T, 1974AnP...486..325T]",https://ui.adsabs.harvard.edu/link_gateway/198...
7,1975AnP...487..383T,"[Treder, H.J.]",Zur unitarisierten Gravitationstheorie mit lan...,1975,Annalen der Physik,AnP,,487,383,400,,,10.1002/andp.19754870509,AA(-),article,26,"[1963ForPh..11...81T, 1973AnP...485..229T, 197...",None
8,1988mqg..book.....V,"[von Borzeszkowski, H.H., Treder, H.J.]",The meaning of quantum gravity.,1988,The meaning of quantum gravity.. H.-H. von Bor...,mqg..book,,20,,,Should the general theory of relativity be qua...,,,AA(-); AB(-),book,20,[],None
9,1985FoPh...15..161T,"[Treder, H. J.]",The planckions as largest elementary particles...,1985,Foundations of Physics,FoPh,2,15,161,166,Planck's units define the limits of super GUT ...,"Elementary Particle, Small Body, Small Test, C...",10.1007/BF00735287,"AA(Einstein-Laboratorium der AdW der DDR, Rosa...",article,11,"[1963ForPh..11...81T, 1982FoPh...12.1113V, 198...",None


---
# Phase 2: Translation

Detect non-English titles and abstracts with fasttext,
then translate them using either OpenRouter (API) or a local HuggingFace model.

### 2.1 Translation Configuration

Choose provider/model and translation target language.
Keep settings explicit so costs and outputs are reproducible.


In [9]:
# ── CONFIGURE ─────────────────────────────────────────
# Provider: 'openrouter' (API; requires OPENROUTER_API_KEY) or 'huggingface' (local)
TRANSLATION_PROVIDER = "openrouter"
TRANSLATION_MODEL = "google/gemini-3-flash-preview"
TRANSLATION_API_KEY = os.getenv("OPENROUTER_API_KEY")
TRANSLATION_MAX_WORKERS = 10

# Path to fasttext language ID model:
# https://dl.fbaipublicfiles.com/fasttext/supervised-models/lid.176.bin
FASTTEXT_MODEL = paths["models"] / "lid.176.bin"
# ──────────────────────────────────────────────────────

### 2.2 Language Detection

Detect language tags for configured text columns.
Only non-target-language rows are translated in the next step.


In [10]:
if START_AT_PHASE <= 2:
    from ads_bib.translate import detect_languages
    
    logger.info("=== Publications ===")
    publications = detect_languages(publications, ["Title", "Abstract"], model_path=FASTTEXT_MODEL)
    
    logger.info("\n=== References ===")
    refs = detect_languages(refs, ["Title", "Abstract"], model_path=FASTTEXT_MODEL)
else:
    logger.info("Skipping Phase 2 Translation (START_AT_PHASE=%s)", START_AT_PHASE)

=== Publications ===
  Title: 183 non-English entries detected
  Abstract: 35 non-English entries detected

=== References ===
  Title: 185 non-English entries detected
  Abstract: 36 non-English entries detected


### 2.3 Translate Non-English Entries

Translate non-English rows and track token/cost metadata.
This harmonizes text fields for downstream NLP and topic modeling.


In [11]:
if START_AT_PHASE <= 2:
    from ads_bib.translate import translate_dataframe

    logger.info("=== Translating Publications ===")
    publications, cost_pubs = translate_dataframe(
        publications, ["Title", "Abstract"],
        provider=TRANSLATION_PROVIDER,
        model=TRANSLATION_MODEL,
        api_key=TRANSLATION_API_KEY,
        max_workers=TRANSLATION_MAX_WORKERS,
        openrouter_cost_mode=OPENROUTER_COST_MODE,
        cost_tracker=tracker,
    )

    logger.info("\n=== Translating References ===")
    refs, cost_refs = translate_dataframe(
        refs, ["Title", "Abstract"],
        provider=TRANSLATION_PROVIDER,
        model=TRANSLATION_MODEL,
        api_key=TRANSLATION_API_KEY,
        max_workers=TRANSLATION_MAX_WORKERS,
        openrouter_cost_mode=OPENROUTER_COST_MODE,
        cost_tracker=tracker,
    )
else:
    logger.info("Skipping Phase 2 Translation (START_AT_PHASE=%s)", START_AT_PHASE)

=== Translating Publications ===
  Title: translating 183 entries with openrouter/google/gemini-3-flash-preview ...


  Title:   0%|          | 0/183 [00:00<?, ?it/s]

HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
HTTP

  Abstract:   0%|          | 0/35 [00:00<?, ?it/s]

HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
HTTP

  Title:   0%|          | 0/185 [00:00<?, ?it/s]

HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
HTTP

KeyboardInterrupt: 

### 2.4 Save Translation Checkpoint

Persist translated data to global cache and run folder for Phase 3 restarts.

In [ ]:
if START_AT_PHASE <= 2:
    from ads_bib._utils.checkpoints import save_translated_checkpoint

    save_translated_checkpoint(
        publications,
        refs,
        cache_dir=paths["cache"],
        run_data_dir=run.paths["data"],
    )

---
# Phase 3: Tokenization

Create `full_text` (Title + Abstract) and tokenize publications with spaCy.
References are intentionally skipped in this phase for runtime stability.

In [ ]:
import os

# ── CONFIGURE ─────────────────────────────────────────
TOKENIZATION_SPACY_MODEL = "en_core_web_md"
TOKENIZATION_BATCH_SIZE = 512
TOKENIZATION_N_PROCESS = min(max((os.cpu_count() or 1) - 1, 1), 8)
TOKENIZATION_DISABLE = ("ner", "parser", "textcat")
# ──────────────────────────────────────────────────────

if START_AT_PHASE <= 3:
    from ads_bib._utils.checkpoints import load_translated_checkpoint
    from ads_bib.tokenize import ensure_spacy_model, tokenize_texts

    if START_AT_PHASE == 3:
        logger.info("Loading translated data from global cache (START_AT_PHASE=3) ...")
        publications, refs = load_translated_checkpoint(
            cache_dir=paths["cache"],
            run_data_dir=run.paths["data"],
        )

    model_to_use, preloaded_nlp = ensure_spacy_model(
        spacy_model=TOKENIZATION_SPACY_MODEL,
        fallback_model="en_core_web_lg",
        spacy_disable=TOKENIZATION_DISABLE,
        auto_download=True,
    )

    logger.info("Tokenizing publications only ...")
    publications = tokenize_texts(
        publications,
        spacy_model=model_to_use,
        nlp=preloaded_nlp,
        batch_size=TOKENIZATION_BATCH_SIZE,
        n_process=TOKENIZATION_N_PROCESS,
        spacy_disable=TOKENIZATION_DISABLE,
    )
    logger.info("refs tokenization skipped by design.")
else:
    logger.info("Skipping Phase 3 Tokenization (START_AT_PHASE=%s)", START_AT_PHASE)

### Checkpoint: Save Phase 3 Results

Persist tokenized publications and translated references for Phase 4 restarts.
This avoids rerunning API-heavy earlier phases.


In [ ]:
if START_AT_PHASE <= 3:
    from ads_bib._utils.checkpoints import save_phase3_checkpoint

    save_phase3_checkpoint(
        publications,
        refs,
        cache_dir=paths["cache"],
        run_data_dir=run.paths["data"],
    )


---
# Phase 4: Author Name Disambiguation (Placeholder)

This step will use an external AND package once it's ready.
For now, the pipeline continues without disambiguation.

In [ ]:
# === AND PLACEHOLDER ===
# Uncomment when the AND package is available:
#
# from and_package import disambiguate
# publications = disambiguate(publications)
# refs = disambiguate(refs)

logger.info("AND step skipped (placeholder). Continuing without disambiguation.")

---
# Phase 5: Topic Modeling & Curation

Use BERTopic to discover thematic clusters, visualize with datamapplot,
then remove unwanted clusters to curate the dataset.

**Important:** Set parameters below based on your dataset size from Phase 1.

### 5.1 Embedding Configuration

Configure embedding provider/model and optional sampling.
These settings strongly affect topic quality and runtime/cost.


In [ ]:
# ── CONFIGURE ─────────────────────────────────────────
# Providers: 'openrouter' (API), 'huggingface_api' (remote via LiteLLM), 'local' (SentenceTransformers)
EMBEDDING_PROVIDER = "openrouter"
EMBEDDING_MODEL = "google/gemini-embedding-001"
EMBEDDING_API_KEY = os.getenv("OPENROUTER_API_KEY")
EMBEDDING_MAX_WORKERS = 10

# For testing: set SAMPLE_SIZE to an integer (e.g. 200). None = full dataset.
SAMPLE_SIZE = None
# ──────────────────────────────────────────────────────

### 5.2 Compute Embeddings

Create or load cached embeddings for `full_text`.
Caching keeps repeated runs fast and reproducible.


In [ ]:
if START_AT_PHASE >= 4:
    from ads_bib._utils.checkpoints import load_phase3_checkpoint

    publications, refs = load_phase3_checkpoint(
        cache_dir=paths["cache"], run_data_dir=run.paths["data"],
    )

In [ ]:
if START_AT_PHASE <= 5:
    from ads_bib.topic_model import compute_embeddings

    df = publications.copy()
    if SAMPLE_SIZE is not None:
        df = df.sample(n=min(SAMPLE_SIZE, len(df)), random_state=RANDOM_SEED).reset_index(drop=True)
        logger.info("SAMPLING: %s documents", f"{len(df):,}")

    documents = df["full_text"].tolist()

    embeddings = compute_embeddings(
        documents,
        provider=EMBEDDING_PROVIDER,
        model=EMBEDDING_MODEL,
        cache_dir=paths["embeddings_cache"],
        max_workers=EMBEDDING_MAX_WORKERS,
        api_key=EMBEDDING_API_KEY,
        openrouter_cost_mode=OPENROUTER_COST_MODE,
        cost_tracker=tracker,
    )
else:
    logger.info("Skipping Phase 5.2 Compute Embeddings (START_AT_PHASE=%s)", START_AT_PHASE)

### 5.3 Dimensionality Reduction Configuration

- Methods: `pacmap` (fast) or `umap` (preferred for hierarchical Toponymy structure).
- Heuristic: smaller/sparser sets use `n_neighbors` ~15; larger/denser sets often benefit from 50-60.
- Use `min_dist=0.0` for 5D clustering vectors and `min_dist=0.1` for 2D visualization.


In [ ]:
# ── CONFIGURE ─────────────────────────────────────────
DIM_REDUCTION_METHOD = "pacmap"  # PaCMAP als lokaler/globaler Kompromiss für großes GRG-Korpus

PARAMS_5D = dict(n_neighbors=60, metric="angular", random_state=RANDOM_SEED)
PARAMS_2D = dict(n_neighbors=60, metric="angular", random_state=RANDOM_SEED)
# ──────────────────────────────────────────────────────

In [ ]:
if START_AT_PHASE <= 5:
    from ads_bib.topic_model import reduce_dimensions

    reduced_5d, reduced_2d = reduce_dimensions(
        embeddings,
        method=DIM_REDUCTION_METHOD,
        params_5d=PARAMS_5D,
        params_2d=PARAMS_2D,
        random_state=RANDOM_SEED,
        cache_dir=paths["dim_reduction_cache"],
        embedding_id=f"{EMBEDDING_PROVIDER}/{EMBEDDING_MODEL}",
    )
else:
    logger.info("Skipping Phase 5.3 Dimensionality Reduction (START_AT_PHASE=%s)", START_AT_PHASE)

### 5.4 Clustering Configuration

**Adjust clustering granularity based on dataset size.**
- `MIN_CLUSTER_SIZE` controls BERTopic/HDBSCAN topic granularity (roughly ~0.1% of docs).
- `BASE_MIN_CLUSTER_SIZE` controls Toponymy micro-cluster granularity.
- `TOPONYMY_LAYER_INDEX` selects the fallback primary layer for visualization.


In [ ]:
# ── CONFIGURE ─────────────────────────────────────────
CLUSTERING_METHOD = "fast_hdbscan"  # Schnellste Variante für 87k

n_docs = len(df)
MIN_CLUSTER_SIZE = max(15, int(n_docs * 0.001)) # ca. 87 bei 87k Dokumenten
BASE_MIN_CLUSTER_SIZE = max(55, int(n_docs * 0.0007))
# ──────────────────────────────────────────────────────

logger.info("Dataset: %s documents", f"{n_docs:,}")
logger.info("  MIN_CLUSTER_SIZE=%s", MIN_CLUSTER_SIZE)
logger.info("  BASE_MIN_CLUSTER_SIZE=%s", BASE_MIN_CLUSTER_SIZE)

CLUSTER_PARAMS = dict(
    min_cluster_size=MIN_CLUSTER_SIZE,
    min_samples=2, # Reduziert Outlier drastisch, da weniger Dichte für Cluster-Zugehörigkeit gefordert wird
    cluster_selection_method="eom",
    cluster_selection_epsilon=0.05, # Absorbiert Randpunkte in bestehende Cluster
)

TOPONYMY_CLUSTER_PARAMS = dict(
    min_clusters=10,
    min_samples=2,
    base_min_cluster_size=BASE_MIN_CLUSTER_SIZE,
)

# EVoC uses a backend-specific constructor in some versions.
# Keep this dict conservative; unsupported keys are filtered in fit_toponymy.
TOPONYMY_EVOC_CLUSTER_PARAMS = dict(
    min_clusters=10,
    min_samples=2,
    base_min_cluster_size=BASE_MIN_CLUSTER_SIZE,
    noise_level=0.35,
    n_neighbors=15,
    n_epochs=35,
)

TOPONYMY_LAYER_INDEX = 1  # Broader Toponymy layer to reduce -1 outliers on large corpora.

### 5.5 Backend & LLM Configuration

Backend matrix:
- `bertopic`: BERTopic on 5D reduced vectors + optional outlier reassignment refresh.
- `toponymy`: Toponymy + `ToponymyClusterer` on 5D reduced vectors.
- `toponymy_evoc`: Toponymy + `EVoCClusterer` on raw embeddings (no 5D clustering vectors).

LLM provider matrix:
- BERTopic: `local`, `huggingface_api`, `openrouter`
- Toponymy / Toponymy+EVoC: `local` (Toponymy HuggingFaceNamer) or `openrouter`

`MIN_DF` guidance:
- small sample runs: often `1`
- full runs: usually `2+`


In [ ]:
# ── CONFIGURE ─────────────────────────────────────────
TOPIC_BACKEND = "bertopic"  # 'bertopic', 'toponymy', or 'toponymy_evoc'

# BERTopic providers: 'local', 'huggingface_api', 'openrouter'
# Toponymy providers: 'local' (HuggingFaceNamer) or 'openrouter'
LLM_PROVIDER = "openrouter"
LLM_MODEL = "google/gemini-3-flash-preview"
LLM_API_KEY = os.getenv("OPENROUTER_API_KEY")

PIPELINE_MODELS = ["POS", "KeyBERT", "MMR"]
PARALLEL_MODELS = ["MMR", "POS", "KeyBERT"]

TOPONYMY_EMBEDDING_MODEL = EMBEDDING_MODEL
TOPONYMY_MAX_WORKERS = 10

# MIN_DF=5 filtert extrem seltene Wörter heraus. Beschleunigt c-TF-IDF bei 87k enorm und spart RAM.
MIN_DF = 5
# ──────────────────────────────────────────────────────

### 5.5b Fit Topic Model

Run the selected backend (BERTopic or Toponymy) on reduced vectors.
This is the most compute/cost-intensive step of the pipeline.

In [ ]:
if START_AT_PHASE <= 5:
    import numpy as np
    from ads_bib.topic_model import fit_bertopic, fit_toponymy, reduce_outliers

    if TOPIC_BACKEND == "bertopic":
        topic_model = fit_bertopic(
            documents,
            reduced_5d,
            llm_provider=LLM_PROVIDER,
            llm_model=LLM_MODEL,
            pipeline_models=PIPELINE_MODELS,
            parallel_models=PARALLEL_MODELS,
            min_df=MIN_DF,
            clustering_method=CLUSTERING_METHOD,
            clustering_params=CLUSTER_PARAMS,
            api_key=LLM_API_KEY,
            openrouter_cost_mode=OPENROUTER_COST_MODE,
            cost_tracker=tracker,
        )

        topics = np.array(topic_model.topics_)
        topics = reduce_outliers(
            topic_model,
            documents,
            topics,
            reduced_5d,
            threshold=0.4,
            llm_provider=LLM_PROVIDER,
            llm_model=LLM_MODEL,
            api_key=LLM_API_KEY,
            openrouter_cost_mode=OPENROUTER_COST_MODE,
            cost_tracker=tracker,
        )
        topic_info = topic_model.get_topic_info()

    elif TOPIC_BACKEND in {"toponymy", "toponymy_evoc"}:
        _cluster_params = (
            TOPONYMY_EVOC_CLUSTER_PARAMS if TOPIC_BACKEND == "toponymy_evoc"
            else TOPONYMY_CLUSTER_PARAMS
        )
        topic_model, topics, topic_info = fit_toponymy(
            documents,
            embeddings,
            reduced_5d,
            backend=TOPIC_BACKEND,
            layer_index=TOPONYMY_LAYER_INDEX,
            llm_provider=LLM_PROVIDER,
            llm_model=LLM_MODEL,
            embedding_model=TOPONYMY_EMBEDDING_MODEL,
            api_key=LLM_API_KEY,
            openrouter_cost_mode=OPENROUTER_COST_MODE,
            max_workers=TOPONYMY_MAX_WORKERS,
            clusterer_params=_cluster_params,
            cost_tracker=tracker,
        )

    else:
        raise ValueError(f"Invalid TOPIC_BACKEND '{TOPIC_BACKEND}'.")

    tracker.log_steps_summary([
        "llm_labeling", "llm_labeling_post_outliers",
        "llm_labeling_toponymy", "llm_labeling_toponymy_evoc",
        "toponymy_embeddings",
    ])
else:
    logger.info("Skipping Phase 5.5b Fit Topic Model (START_AT_PHASE=%s)", START_AT_PHASE)

### 5.5c Build Topic DataFrame

Merge topic assignments, 2D coordinates, and embeddings into the main DataFrame.

In [ ]:
if START_AT_PHASE <= 5:
    from ads_bib.topic_model import build_topic_dataframe

    df = build_topic_dataframe(
        df,
        topic_model,
        topics,
        reduced_2d,
        embeddings,
        topic_info=topic_info,
    )
    display(topic_info)
else:
    logger.info("Skipping Phase 5.5c Build Topic DataFrame (START_AT_PHASE=%s)", START_AT_PHASE)

### 5.6 Visualize Topics

Render an interactive topic map from 2D coordinates and topic labels.
Use this view to inspect cluster semantics before curation.


In [ ]:
if START_AT_PHASE <= 5:
    from ads_bib.visualize import create_topic_map

    plot = create_topic_map(
        df,
        title="ADS Bibliometric Map",
        subtitle=f"Topics labeled with {LLM_PROVIDER}/{LLM_MODEL}",
        dark_mode=True,
        output_path=run.paths["plots"] / "topic_map.html",
    )
    # plot  # uncomment to display inline
else:
    logger.info("Skipping Phase 5.6 Visualize Topics (START_AT_PHASE=%s)", START_AT_PHASE)

### 5.7 Curate Dataset

Review the cluster summary, then remove clusters that don't fit your research question.

In [ ]:
if START_AT_PHASE <= 5:
    from ads_bib.curate import get_cluster_summary, remove_clusters

    display(get_cluster_summary(df))
else:
    logger.info("Skipping Phase 5.7 Curate Dataset (START_AT_PHASE=%s)", START_AT_PHASE)

### Checkpoint: Save Curated Dataset

Save the curated topic dataset as the handoff for citation analysis.
This provides a stable artifact for Phase 6 and external reuse.


In [ ]:
if START_AT_PHASE <= 5:
    from ads_bib._utils.io import save_parquet

    # Ensure References is list type for Parquet
    df["References"] = df["References"].apply(lambda x: x if isinstance(x, list) else [])

    save_parquet(df, run.paths["data"] / "curated_dataset.parquet")
    logger.info("Curated dataset: %s documents", f"{len(df):,}")
else:
    logger.info("Skipping Checkpoint: Save Curated Dataset (START_AT_PHASE=%s)", START_AT_PHASE)

In [ ]:
if START_AT_PHASE <= 5:
    from ads_bib._utils.io import load_parquet

    df = load_parquet(run.paths["data"] / "curated_dataset.parquet")
    display(df.head(10))
else:
    logger.info("Skipping Checkpoint: Load Curated Dataset (START_AT_PHASE=%s)", START_AT_PHASE)

---
# Phase 6: Citation Networks

Compute citation networks from the curated dataset and export
to GEXF (Gephi), Graphology JSON (Sigma.js), CSV, and/or Web of Science format.

### 6.1 Citation Configuration

Set network metrics, thresholds, filters, and export formats.
These parameters define which citation structures are constructed.

In [ ]:
# ── CONFIGURE ─────────────────────────────────────────
CITE_METRICS = ["direct", "co_citation", "bibliographic_coupling", "author_co_citation"]
MIN_COUNTS = {
    "direct": 1,
    "co_citation": 1,
    "bibliographic_coupling": 1,
    "author_co_citation": 1,
}
AUTHORS_FILTER = None
OUTPUT_FORMAT = "gexf"  # 'gexf', 'graphology', 'csv', 'all'
CLUSTERS_TO_REMOVE = []
# ──────────────────────────────────────────────────────

# Snapshot the full configuration once all phase configs are defined.
run.save_config(globals())

### 6.2 Build Node Table & Compute Networks

Build node metadata and compute selected citation networks.
Outputs are exported for Gephi/Sigma/CSV workflows.


In [ ]:
if START_AT_PHASE <= 6:
    if START_AT_PHASE == 6:
        from ads_bib._utils.checkpoints import load_phase3_checkpoint

        publications, refs = load_phase3_checkpoint(
            cache_dir=paths["cache"], run_data_dir=run.paths["data"],
        )

    from ads_bib.citations import (
        build_all_nodes,
        build_citation_inputs_from_publications,
        export_wos_format,
        process_all_citations,
    )

    bibcodes, references = build_citation_inputs_from_publications(publications)
    all_nodes = build_all_nodes(publications, refs)

    results = process_all_citations(
        bibcodes=bibcodes,
        references=references,
        publications=publications,
        ref_df=refs,
        all_nodes=all_nodes,
        metrics=CITE_METRICS,
        min_counts=MIN_COUNTS,
        authors_filter=AUTHORS_FILTER,
        output_format=OUTPUT_FORMAT,
        output_dir=run.paths["data"],
    )
else:
    logger.info("Skipping Phase 6 Citation Networks (START_AT_PHASE=%s)", START_AT_PHASE)

### 6.3 Export Web of Science Format

Export the curated corpus in WOS text format.
This supports downstream tooling such as CiteSpace/VOSviewer.


In [ ]:
suffix = "_filtered" if AUTHORS_FILTER else ""
export_wos_format(
    publications, refs,
    output_path=run.paths["data"] / f"download_wos_export{suffix}.txt",
)

---
# Summary

Final dataset statistics and output file listing.

In [ ]:
logger.info("=" * 60)
logger.info("PIPELINE COMPLETE")
logger.info("=" * 60)
logger.info("Publications:     %s", f"{len(publications):,}")
logger.info("References:       %s", f"{len(refs):,}")
logger.info("Curated dataset:  %s", f"{len(df):,}")
logger.info("Topics found:     %s", df["topic_id"].nunique())
logger.info("")
logger.info("Output files:")
for root, dirs, files in os.walk(run.paths["root"]):
    for f in sorted(files):
        fpath = Path(root) / f
        size_mb = fpath.stat().st_size / 1024 / 1024
        logger.info("  %s (%.1f MB)", fpath.relative_to(run.paths["root"]), size_mb)

logger.info("")
logger.info(tracker.compact_summary())